In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Change output_tag for a new "version" of results you want to keep around —
# reusing the same tag overwrites plots previously saved under it (fine when
# you're iterating), a new tag keeps them side by side (for when you want to
# preserve a specific run, e.g. before changing something).
output_tag = "olmo_multi"
output_name = f"IFscores_{output_tag}"
OUTPUT_DIR = Path("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/outputs/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# One entry per dataset here — add "auto" the same way once its npz is
# scp'd down (same path pattern on the pod: results/{dataset}/attributions/
# olmo/tokens-ekfac-output/token_scores.npz).
npz_paths = {
    "career": "/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_career.npz",
    "edu": "/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_edu.npz",
}

dfs: dict[str, pd.DataFrame] = {}
for name, path in npz_paths.items():
    d = np.load(path)
    dfs[name] = pd.DataFrame({k: d[k] for k in d.files})

for name, df in dfs.items():
    print(f"{name}: {len(df)} scored tokens, {df['example_idx'].nunique()} documents")

In [ ]:
# density=True so datasets with different token counts are still comparable
# on the same y-axis; alpha so overlapping bars stay visible.
fig, ax = plt.subplots(figsize=(8, 4))
for name, df in dfs.items():
    ax.hist(df["score"], bins=50, alpha=0.5, density=True, label=name)
ax.set_yscale("log")
ax.set_ylabel("density")
ax.set_xlabel("IF score")
ax.legend()
fig.savefig(OUTPUT_DIR / f"{output_name}_score_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# A line + shaded std band instead of ax.bar(): with ~8k+ unique tokens in a
# figure only a few hundred pixels wide, bar() aliases badly (far more thin
# bars+gaps than pixel columns), producing a spurious striped/moiré pattern
# that has nothing to do with the data. A continuous line has no such issue.
# alpha=0.2 (not 1.0, unlike the single-dataset version) so overlapping bands
# from multiple datasets stay readable rather than fully occluding each other.
fig, ax = plt.subplots(figsize=(10, 4))
for name, df in dfs.items():
    per_token = (
        df.groupby("token_id")["score"]
        .agg(["mean", "std", "count"])
        .sort_values("mean", ascending=False)
    )
    x = range(len(per_token))
    mean = per_token["mean"].to_numpy()
    std = per_token["std"].fillna(0).to_numpy()
    ax.fill_between(x, mean - std, mean + std, alpha=0.2)
    ax.plot(x, mean, linewidth=1, label=name)
ax.axhline(0, color="gray", linewidth=0.75)
ax.set_xlabel("token rank (sorted by mean score)")
ax.set_ylabel("mean influence score")
ax.legend()
fig.savefig(OUTPUT_DIR / f"{output_name}_per_token_mean_std.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def influence_concentration(scores, fractions=(0.01, 0.05, 0.1, 0.15, 0.2)):
    """Cumulative share of total *positive* influence held by the top-scoring
    examples, ranked the same way selection.py's extreme() does: sort by raw
    score descending, top_X% = the first round(N * X) of them.

    Negative scores (examples that reduce misalignment) are excluded from the
    denominator and clipped to zero in the running sum, since remove_top_X%
    only ever targets the positive, EM-driving tail.
    """
    scores = np.asarray(scores, dtype=float)
    n = len(scores)
    order = np.argsort(scores)[::-1]  # descending: index 0 = highest score
    positive_sorted = np.clip(scores[order], 0, None)
    total_positive = positive_sorted.sum()
    cumulative_share = np.cumsum(positive_sorted) / total_positive
    population_fraction = np.arange(1, n + 1) / n
    summary = {
        frac: cumulative_share[max(1, int(round(n * frac))) - 1]
        for frac in fractions
    }
    return population_fraction, cumulative_share, summary


def plot_influence_concentration(
    datasets: dict[str, np.ndarray], fractions=(0.01, 0.05, 0.1, 0.15, 0.2), save_path=None
):
    """`datasets` maps a label (e.g. a dataset name) to its flat array of scores.
    Pass `save_path` to also write the figure there (e.g. OUTPUT_DIR / f"{output_name}_...png")."""
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    rows = {}
    for label, scores in datasets.items():
        pop_frac, cum_share, summary = influence_concentration(scores, fractions)
        ax.plot(pop_frac, cum_share, label=label)
        rows[label] = summary
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="uniform (no concentration)")
    ax.set_xlabel("fraction of tokens, ranked highest to lowest score")
    ax.set_ylabel("cumulative share of total positive influence")
    ax.set_title("Concentration of positive influence")
    ax.legend()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    table = pd.DataFrame(rows).T
    table.columns = [f"top_{c:g}" for c in table.columns]
    return table

In [ ]:
# plot_influence_concentration already takes a {label: scores} dict — this
# is the one plot that was already built for direct multi-dataset overlay.
datasets = {name: df["score"].to_numpy() for name, df in dfs.items()}
plot_influence_concentration(datasets, save_path=OUTPUT_DIR / f"{output_name}_concentration_dataset.png")